In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score

# Load raw data
train_df = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

# Save test PassengerIds
passenger_ids = test_df['PassengerId']

# Combine to ensure identical feature scaling and categories
df = pd.concat([train_df.drop('Survived', axis=1), test_df], axis=0).reset_index(drop=True)

# 1. Ticket Grouping (Extremely Powerful)
# Find how many passengers share the exact same Ticket number (Group Size)
df['GroupSize'] = df.groupby('Ticket')['Ticket'].transform('count')

# Calculate the actual fare paid per individual
df['FarePerPerson'] = df['Fare'] / df['GroupSize']

# 2. Extract Title from Name
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
df['Title'] = df['Title'].replace(rare_titles, 'Rare')
df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# 3. Family Size & Alone Flags
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# 4. Extract Deck from Cabin
df['Deck'] = df['Cabin'].apply(lambda x: x[0] if isinstance(x, str) else 'U')

# 5. Safe Missing Value Imputation
# Impute Age using Pclass + Sex groups
df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['FarePerPerson'] = df['FarePerPerson'].fillna(df['FarePerPerson'].median())

# Drop non-generalizable ID features
df = df.drop(['Name', 'Ticket', 'Cabin'], axis=1)

# 6. One-Hot Encode Categorical Variables
df = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Title', 'Deck'], drop_first=True)

# Split back to Train and Test sets
X_train = df.iloc[:len(train_df)].copy()
X_test = df.iloc[len(train_df):].copy()
y_train = train_df['Survived']

print("Features processed successfully!")
print("Train Set Shape:", X_train.shape)

Features processed successfully!
Train Set Shape: (891, 25)


In [2]:
# Standardize features (necessary for the SVC component)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define 1: Support Vector Classifier (SVC)
# Note: probability=True is necessary for soft voting
svc_model = SVC(C=1.0, kernel='rbf', gamma='scale', probability=True, random_state=42)

# Define 2: Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=150, max_depth=5, min_samples_split=4, random_state=42)

# Define 3: Gradient Boosting Classifier
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)

# Combine the three models into a Soft Voting Ensemble
ensemble_model = VotingClassifier(
    estimators=[
        ('svc', svc_model), 
        ('rf', rf_model), 
        ('gb', gb_model)
    ],
    voting='soft' # Takes the average of predicted probabilities
)

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

# Perform Cross-Validation manually to evaluate stability
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_scaled, y_train)):
    X_tr, y_tr = X_train_scaled[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train_scaled[val_idx], y_train.iloc[val_idx]
    
    ensemble_model.fit(X_tr, y_tr)
    preds = ensemble_model.predict(X_va)
    acc = accuracy_score(y_va, preds)
    scores.append(acc)
    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

print(f"\nMean Cross-Validation Accuracy: {np.mean(scores):.4f}")

# Train final ensemble model on the entire dataset
ensemble_model.fit(X_train_scaled, y_train)

Fold 1 Accuracy: 0.8492
Fold 2 Accuracy: 0.8371
Fold 3 Accuracy: 0.8315
Fold 4 Accuracy: 0.8202
Fold 5 Accuracy: 0.8483

Mean Cross-Validation Accuracy: 0.8372


VotingClassifier(estimators=[('svc', SVC(probability=True, random_state=42)),
                             ('rf',
                              RandomForestClassifier(max_depth=5,
                                                     min_samples_split=4,
                                                     n_estimators=150,
                                                     random_state=42)),
                             ('gb',
                              GradientBoostingClassifier(learning_rate=0.05,
                                                         random_state=42))],
                 voting='soft')

In [4]:
# Predict using the final ensemble
test_predictions = ensemble_model.predict(X_test_scaled)

# Prepare submission file
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': test_predictions
})

# Save to working directory
submission.to_csv('submission.csv', index=False)
print("Updated 'submission.csv' generated with advanced features and ensemble voting!")

Updated 'submission.csv' generated with advanced features and ensemble voting!
